# 01 · Precompute demo fields, Engines 1–3

Builds the per-field inputs for Engines 1–3 (handoff `docs/handoff_v2.md` §8, §14) and writes them to `data/processed/<field_id>/`. The Streamlit app reads only these files, so the demo never depends on a live API call.

| Step | Handoff §14 | Source | Login |
|---|---|---|---|
| 1 Demo fields, inward buffer, pixel count (V1 pre-check) | 1 | `pipeline/demo_fields.geojson` | – |
| 2 Crop sequence 2016–2025 | 4 | USDA CDL via CroplandCROS | – |
| 3 Soil properties | 10 | USDA SSURGO via Soil Data Access | – |
| 4 Field slope (V12) | 10 | NASADEM (NASA SRTM reprocessing); 3DEP reference | – |
| 5 Daily weather, root-zone wetness, capped GDD (V10) | 7–8 | NASA POWER 1981–2025 | – |
| 6 AppEEARS tasks: HLS NDVI + Fmask, SMAP L4, ECOSTRESS ESI | 2, 8 | NASA AppEEARS | Earthdata |
| 7 HLS per-date field NDVI, V1/V2 | 2–3 | NASA HLS v2.0 (Landsat + Sentinel-2) | Earthdata |
| 9 Engine 1: phenometrics, living cover, spring/autumn green | 5–6 | from step 7 | – |
| 10 Engine 2: sensitive windows, stress flags vs base rate, vigour, scenarios | 7–9 | Engine 1 + POWER (+ SMAP L4) | – |
| 11 Engine 3: rotation ratings with evidence and confidence | 10 | Engines 1–2, soil, slope, crop catalogue | – |

Run from the repo root kernel, or any kernel: the first cell puts the repo on `sys.path`.

In [1]:
import json, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
REPO = Path.cwd() if (Path.cwd() / "pipeline").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

import pandas as pd
from pipeline import fields
from pipeline.config import PROCESSED, RAW, SEASONS
from pipeline.gdd import gdd_day
from pipeline.provenance import DATASETS, run_stamp, sourced
from pipeline.sources import appeears, cdl, dem, hls, power, ssurgo

pd.set_option("display.width", 160)
STAMP = run_stamp()
STAMP

'2026-11-14T09:11:06Z'

## 1 · Demo fields

Selected on 14 Nov from a 10 × 10 km window east of Ames: blocks whose CDL class strictly alternates corn/soybean in every season 2016–2025, reduced to the largest inscribed rectangle and checked on NAIP 2025 (see `pipeline/fields.py`).

In [2]:
F = fields.load_demo_fields()
pd.DataFrame([{k: f[k] for k in ("id", "county", "area_ha", "interior_pixels_approx", "corn_years", "naip_note")} for f in F])

,id,county,area_ha,interior_pixels_approx,corn_years,naip_note
0,field_a,"Story County, Iowa",50.6,466,even,Wet spots (prairie potholes)
1,field_b,"Story County, Iowa",45.6,420,odd,Uniform
2,field_c,"Story County, Iowa",37.9,342,odd,Light-coloured knolls (likely eroded)


## 2 · Crop sequence (CDL)

Majority CDL class over the field interior each season and its share of the interior pixels. CDL is used directly, so V5 (crop inference) is skipped, as the handoff allows.

In [3]:
seq = {f["id"]: [cdl.field_stats(f["interior"], y) for y in SEASONS] for f in F}
pd.DataFrame({fid: [f"{d['name']} ({d['share']:.0%})" for d in rows] for fid, rows in seq.items()}, index=list(SEASONS))

,field_a,field_b,field_c
2016,corn (100%),soybeans (100%),soybeans (100%)
2017,soybeans (100%),corn (100%),corn (100%)
2018,corn (100%),soybeans (100%),soybeans (100%)
2019,soybeans (100%),corn (100%),corn (100%)
2020,corn (100%),soybeans (100%),soybeans (100%)
2021,soybeans (100%),corn (100%),corn (100%)
2022,corn (100%),soybeans (100%),soybeans (100%)
2023,soybeans (99%),corn (100%),corn (100%)
2024,corn (100%),soybeans (100%),soybeans (100%)
2025,soybeans (100%),corn (100%),corn (100%)


## 3 · Soil (SSURGO)

Area-weighted over map units intersecting the field. `aws0100wta` is available water storage in the top 100 cm (cm); `om_r` and `kwfact` are from the dominant component's surface horizon.

In [4]:
soils, soil_units = {}, {}
for f in F:
    mu, summary = ssurgo.field_soils(f["geometry"])
    soils[f["id"]], soil_units[f["id"]] = summary, mu
pd.DataFrame(soils).T

,aws0100wta,slopegradwta,om_r,kwfact,muname,drclassdcd,hydgrpdcd,dominant_share,n_map_units
field_a,18.566,1.792,6.093,0.274,"Webster clay loam, Bemis moraine, 0 to 2 perce...",Poorly drained,C/D,0.466,5
field_b,18.67,1.607,6.312,0.276,"Webster clay loam, Bemis moraine, 0 to 2 perce...",Poorly drained,C/D,0.244,6
field_c,18.451,3.183,5.177,0.274,"Webster clay loam, Bemis moraine, 0 to 2 perce...",Well drained,C/D,0.262,7


In [5]:
for fid, mu in soil_units.items():
    print(fid); print(mu[["muname", "share", "drclassdcd", "aws0100wta", "slopegradwta"]].head(5).to_string(index=False)); print()

field_a
                                                 muname    share              drclassdcd  aws0100wta  slopegradwta
Webster clay loam, Bemis moraine, 0 to 2 percent slopes 0.465567          Poorly drained       18.15           1.0
                   Nicollet loam, 1 to 3 percent slopes 0.371456 Somewhat poorly drained       19.04           2.1
     Clarion loam, Bemis moraine, 2 to 6 percent slopes 0.154290            Well drained       18.64           3.5
  Harps clay loam, Bemis moraine, 0 to 2 percent slopes 0.006289          Poorly drained       18.93           1.0
         Okoboji silty clay loam, 0 to 1 percent slopes 0.002397     Very poorly drained       20.33           0.2

field_b
                                                  muname    share              drclassdcd  aws0100wta  slopegradwta
 Webster clay loam, Bemis moraine, 0 to 2 percent slopes 0.243907          Poorly drained       18.15           1.0
      Clarion loam, Bemis moraine, 2 to 6 percent slopes 0.19

## 4 · Field slope (NASADEM) and V12

Raw 30 m SRTM-family slope is inflated by elevation noise on flat ground, so slope is computed after a Gaussian filter (σ = 1.5 px). **V12 (proposed):** smoothed NASADEM mean slope within ±0.5 percentage points of USGS 3DEP 10 m lidar on every field, and the same steepest field. (A full ranking test is too strict when fields differ by less than ~0.1 pp.)

In [6]:
terrain = {}
for f in F:
    s = dem.field_slope(f["geometry"], f["interior"])
    s["ref_3dep_10m_pct"] = dem.reference_slope_3dep(f["geometry"], f["interior"])
    s["ssurgo_slopegradwta_pct"] = soils[f["id"]]["slopegradwta"]
    terrain[f["id"]] = s
T = pd.DataFrame(terrain).T[["mean_slope_pct", "mean_slope_pct_unsmoothed", "ref_3dep_10m_pct", "ssurgo_slopegradwta_pct", "p90_slope_pct", "relief_m"]]
T["v12_within_0.5pp"] = (T.mean_slope_pct.astype(float) - T.ref_3dep_10m_pct.astype(float)).abs() <= 0.5
v12_steepest = T.mean_slope_pct.astype(float).idxmax() == T.ref_3dep_10m_pct.astype(float).idxmax()
print("V12 all within 0.5 pp:", bool(T["v12_within_0.5pp"].all()), "| same steepest field:", v12_steepest)
T

V12 all within 0.5 pp: True | same steepest field: True


,mean_slope_pct,mean_slope_pct_unsmoothed,ref_3dep_10m_pct,ssurgo_slopegradwta_pct,p90_slope_pct,relief_m,v12_within_0.5pp
field_a,1.72,4.17,1.51,1.792,3.14,10.0,True
field_b,1.62,3.76,1.55,1.607,2.74,14.0,True
field_c,1.99,3.9,2.03,3.183,3.52,13.0,True


## 5 · NASA POWER daily, 1981–2025, and capped GDD

One POWER cell (~0.5°) per field centroid. `GWETROOT` (root-zone soil wetness, 0–1) comes from NASA GMAO MERRA-2: modelled, regional. V10 unit tests live in `tests/test_gdd.py`.

In [7]:
weather = {}
for f in F:
    c = f["geometry"].centroid
    w = power.fetch(c.y, c.x)
    w["gdd_c"] = gdd_day(w.T2M_MAX, w.T2M_MIN)
    weather[f["id"]] = w
    print(f["id"], "POWER point [lon, lat, cell elev m]", w.attrs["power_cell"], "|", len(w), "days", w.index.min().date(), "to", w.index.max().date(), "| missing", int(w.isna().sum().sum()))
w = weather[F[0]["id"]]
w[w.index.month.isin([5, 6, 7, 8, 9])].groupby(w[w.index.month.isin([5, 6, 7, 8, 9])].index.year).agg(gdd_may_sep=("gdd_c", "sum"), rain_mm=("PRECTOTCORR", "sum"), gwetroot_mean=("GWETROOT", "mean")).loc[2016:].round(2)

field_a POWER point [lon, lat, cell elev m] [-93.431, 42.04, 301.38] | 16436 days 1981-01-01 to 2025-12-31 | missing 0
field_b POWER point [lon, lat, cell elev m] [-93.411, 42.06, 301.38] | 16436 days 1981-01-01 to 2025-12-31 | missing 0
field_c POWER point [lon, lat, cell elev m] [-93.451, 42.103, 310.22] | 16436 days 1981-01-01 to 2025-12-31 | missing 0


,gdd_may_sep,rain_mm,gwetroot_mean
date,,,
2016,1654.21,604.46,0.59
2017,1670.56,345.93,0.53
2018,1770.31,740.96,0.57
2019,1567.67,566.30,0.70
2020,1577.97,456.22,0.58
2021,1790.18,357.19,0.49
2022,1746.20,371.79,0.52
2023,1783.50,302.48,0.49
2024,1687.68,593.96,0.56


## 6 · AppEEARS tasks

Submitted on 14 Nov (HLS split by field × sensor × period to stay under AppEEARS' file cap). Re-running this cell is safe: tasks are found by name and not resubmitted.

In [8]:
reg_path = RAW / "appeears_tasks.json"
registry = json.loads(reg_path.read_text()) if reg_path.exists() else {}
status = pd.DataFrame([{"task": n, "task_id": t, "status": appeears.status(t).get("status")} for n, t in registry.items()])
status.groupby("status").size()

status
done          11
pending        5
processing     1
dtype: int64

In [9]:
# Download finished tasks (skips files already present)
for r in status[status.status == "done"].itertuples():
    appeears.download(r.task_id, RAW / "appeears" / r.task, patterns=("*.tif", "*.csv", "*.json"))
sorted(p.name for p in (RAW / "appeears").glob("*")) if (RAW / "appeears").exists() else []

['fieldshift_hls_l30_field_a_2016_2020_v1',
 'fieldshift_hls_l30_field_a_2021_2025_v1',
 'fieldshift_hls_l30_field_b_2016_2020_v1',
 'fieldshift_hls_l30_field_b_2021_2025_v1',
 'fieldshift_hls_l30_field_c_2016_2020_v1',
 'fieldshift_hls_l30_field_c_2021_2025_v1',
 'fieldshift_hls_s30_field_a_2016_2020_v1',
 'fieldshift_hls_s30_field_a_2021_2025_v1',
 'fieldshift_hls_s30_field_b_2016_2020_v1',
 'fieldshift_hls_s30_field_b_2021_2025_v1',
 'fieldshift_hls_s30_field_c_2016_2020_v1']

## 7 · HLS per-date field NDVI (V1, V2)

V1: a date is kept if ≥ 50 valid interior pixels remain after Fmask. V2: seasons with < 8 valid dates April–October are flagged low confidence, not dropped.

In [10]:
obs = {}
for f in F:
    dirs = sorted((RAW / "appeears").glob(f"fieldshift_hls_*_{f['id']}_*")) if (RAW / "appeears").exists() else []
    parts = [hls.field_series(d, f["interior"]) for d in dirs]
    parts = [p for p in parts if not p.empty]
    if parts:
        obs[f["id"]] = pd.concat(parts).sort_values("date").reset_index(drop=True)
        o = obs[f["id"]]
        print(f["id"], len(o), "acquisitions;", int(o.passes_v1.sum()), "pass V1;", "sensors", o.sensor.value_counts().to_dict())
    else:
        print(f["id"], "HLS not downloaded yet:", len(dirs), "task folders")

field_a 851 acquisitions; 566 pass V1; sensors {'S30': 572, 'L30': 279}


field_b 830 acquisitions; 547 pass V1; sensors {'S30': 551, 'L30': 279}


field_c 486 acquisitions; 302 pass V1; sensors {'L30': 279, 'S30': 207}


In [11]:
if obs:
    v2 = pd.concat({fid: hls.season_density(o) for fid, o in obs.items()}, axis=1)
    v2.columns = [c[0] for c in v2.columns]
    display((v2 >= 8).replace({True: "ok", False: "LOW"}).astype(str) + " (" + v2.astype("Int64").astype(str) + ")")

,field_a,field_b,field_c
date,,,
2016,ok (18),ok (16),ok (19)
2017,ok (19),ok (20),ok (17)
2018,ok (39),ok (37),ok (35)
2019,ok (34),ok (36),ok (33)
2020,ok (39),ok (35),ok (32)
2021,ok (47),ok (50),ok (11)
2022,ok (52),ok (45),ok (20)
2023,ok (51),ok (49),ok (18)
2024,ok (53),ok (58),ok (23)


## 8 · Write processed outputs

`field_static.json` holds every non-time-series value with its source; time series go to Parquet.

In [12]:
for f in F:
    fid = f["id"]; out = PROCESSED / fid; out.mkdir(parents=True, exist_ok=True)
    static = {
        "field_id": fid, "generated": STAMP,
        "geometry": f["geometry"].__geo_interface__,
        "area_ha": sourced(f["area_ha"], "field polygon (Albers EPSG:5070)"),
        "interior_pixels_approx": sourced(f["interior_pixels_approx"], "field polygon, 30 m inward buffer"),
        "crop_sequence": [sourced(d["name"], DATASETS["cdl"], str(d["year"]), share=d["share"], cdl_class=d["class"]) for d in seq[fid]],
        "soil": {k: sourced(v, DATASETS["ssurgo"]) for k, v in soils[fid].items()},
        "terrain": {k: sourced(v, DATASETS["nasadem"]) for k, v in terrain[fid].items() if k not in ("ref_3dep_10m_pct", "ssurgo_slopegradwta_pct")}
                   | {"ref_3dep_10m_pct": sourced(terrain[fid]["ref_3dep_10m_pct"], "USGS 3DEP 10 m (validation reference only)")},
        "power_point_lon_lat_elev": weather[fid].attrs["power_cell"],
    }
    (out / "field_static.json").write_text(json.dumps(static, indent=1, default=str))
    weather[fid].to_parquet(out / "power_daily.parquet")
    if fid in obs:
        obs[fid].to_parquet(out / "hls_obs.parquet")
sorted(str(p.relative_to(REPO)) for p in PROCESSED.rglob("*") if p.is_file())

['data/processed/field_a/engine1.json',
 'data/processed/field_a/engine2.json',
 'data/processed/field_a/field_static.json',
 'data/processed/field_a/hls_obs.parquet',
 'data/processed/field_a/ndvi_smooth.parquet',
 'data/processed/field_a/power_daily.parquet',
 'data/processed/field_b/engine1.json',
 'data/processed/field_b/engine2.json',
 'data/processed/field_b/field_static.json',
 'data/processed/field_b/hls_obs.parquet',
 'data/processed/field_b/ndvi_smooth.parquet',
 'data/processed/field_b/power_daily.parquet',
 'data/processed/field_c/engine1.json',
 'data/processed/field_c/engine2.json',
 'data/processed/field_c/field_static.json',
 'data/processed/field_c/hls_obs.parquet',
 'data/processed/field_c/ndvi_smooth.parquet',
 'data/processed/field_c/power_daily.parquet']

## 9 · Engine 1: Field History (handoff §8.1)

Smoothed daily NDVI (weighted Whittaker, one robust pass), green-up / peak / decline per season, living-cover and no-living-cover days per cover year, off-season living days, spring-green and autumn-green signals. Thresholds are provisional; the no-living-cover headline is shown at 0.35 / 0.40 / 0.45 so the team can see how much it depends on the choice. Seasons are analysed only where HLS data has been downloaded.

In [13]:
from pipeline import engine1 as E1
e1_results = {}
for f in F:
    fid = f["id"]
    if fid not in obs:
        print(fid, "no HLS yet"); continue
    o = obs[fid]
    yrs = sorted(set(o.date.dt.year) & set(SEASONS))
    crops = {d["year"]: d["name"] for d in seq[fid]}
    seasons_e1, smooth = E1.run(o, crops, yrs)
    sens = {f"{th:.2f}": E1.summarise(E1.run(o, crops, yrs, threshold=th)[0])["no_living_cover_months_per_year"] for th in (0.35, 0.40, 0.45)}
    cov = {s: [int(g.date.dt.year.min()), int(g.date.dt.year.max())] for s, g in o.groupby("sensor")}
    e1_results[fid] = {"seasons": seasons_e1, "smooth": smooth, "summary": E1.summarise(seasons_e1), "sensitivity": sens, "coverage": cov}
    print(fid, "coverage", cov, "| summary", e1_results[fid]["summary"], "| months/yr by threshold", sens)
if e1_results:
    fid0 = next(iter(e1_results))
    display(pd.DataFrame(E1.to_records(e1_results[fid0]["seasons"]))[["season", "crop", "greenup_date", "peak_date", "peak_ndvi", "decline_date", "living_cover_end", "living_cover_days", "off_season_living_days", "no_living_cover_share", "spring_green", "autumn_green_obs", "confidence"]])

field_a coverage {'L30': [2016, 2025], 'S30': [2016, 2025]} | summary {'complete_cover_years': 9, 'mean_no_living_cover_share': 0.704, 'no_living_cover_months_per_year': 8.5, 'mean_off_season_living_days': 5.0, 'spring_green_seasons': 0, 'spring_assessable_seasons': 9, 'autumn_green_seasons': 1} | months/yr by threshold {'0.35': 8.2, '0.40': 8.5, '0.45': 8.7}
field_b coverage {'L30': [2016, 2025], 'S30': [2016, 2025]} | summary {'complete_cover_years': 9, 'mean_no_living_cover_share': 0.687, 'no_living_cover_months_per_year': 8.2, 'mean_off_season_living_days': 21.0, 'spring_green_seasons': 1, 'spring_assessable_seasons': 9, 'autumn_green_seasons': 4} | months/yr by threshold {'0.35': 7.8, '0.40': 8.2, '0.45': 8.6}


field_c coverage {'L30': [2016, 2025], 'S30': [2016, 2020]} | summary {'complete_cover_years': 9, 'mean_no_living_cover_share': 0.697, 'no_living_cover_months_per_year': 8.4, 'mean_off_season_living_days': 0.0, 'spring_green_seasons': 0, 'spring_assessable_seasons': 8, 'autumn_green_seasons': 0} | months/yr by threshold {'0.35': 8.1, '0.40': 8.4, '0.45': 8.6}


,season,crop,greenup_date,peak_date,peak_ndvi,decline_date,living_cover_end,living_cover_days,off_season_living_days,no_living_cover_share,spring_green,autumn_green_obs,confidence
0,2016,corn,2016-06-15,2016-07-18,0.818,2016-08-17,2016-08-28,82,0,0.660,none detected,3,moderate
1,2017,soybeans,2017-06-22,2017-08-08,0.937,2017-09-19,2017-09-30,151,43,0.621,insufficient data,0,moderate
2,2018,corn,2018-06-19,2018-07-24,0.875,2018-09-11,2018-09-21,102,0,0.713,none detected,0,high
3,2019,soybeans,2019-07-08,2019-08-17,0.933,2019-09-21,2019-10-01,96,0,0.744,none detected,0,high
4,2020,corn,2020-07-01,2020-07-31,0.747,2020-09-13,2020-09-22,88,0,0.754,none detected,0,high
5,2021,soybeans,2021-06-11,2021-07-13,0.935,2021-09-05,2021-09-15,109,0,0.696,none detected,0,high
6,2022,corn,2022-06-27,2022-07-29,0.893,2022-09-14,2022-09-24,97,0,0.741,none detected,0,high
7,2023,soybeans,2023-06-19,2023-08-17,0.942,2023-09-20,2023-09-30,114,0,0.693,none detected,0,high
8,2024,corn,2024-07-01,2024-08-02,0.905,2024-10-04,2024-10-14,114,0,0.700,none detected,0,high
9,2025,soybeans,2025-06-15,2025-07-29,0.941,2025-09-13,2025-09-24,111,0,0.678,none detected,0,high


In [14]:
for fid, r in e1_results.items():
    out = PROCESSED / fid
    src = " + ".join(DATASETS["hls_l30" if s == "L30" else "hls_s30"] + f" {a}-{b}" for s, (a, b) in r["coverage"].items())
    doc = {
        "field_id": fid, "generated": STAMP, "engine": "Engine 1: Field History (handoff §8.1)",
        "source": src,
        "method": {"cover_threshold_ndvi": E1.COVER_THRESHOLD, "spring_green_ndvi": E1.SPRING_GREEN, "whittaker_lambda": E1.WHITTAKER_LAMBDA,
                   "support_days": E1.SUPPORT_DAYS, "greenup": "50% of spring amplitude (as MSLSP30NA 50PCGI)", "decline": "50% of autumn amplitude",
                   "living_cover": "smoothed NDVI >= threshold with a clear observation >= threshold within support_days; any green vegetation",
                   "end_of_living_cover": "first day after peak below threshold (senescence, not harvest)",
                   "spring_green": ">= 2 clear obs >= spring_green_ndvi, 1 Mar to min(10 May, green-up - 30 d); needs >= 2 clear obs 15 Mar-30 Apr"},
        "seasons": [rec | {"source": src} for rec in E1.to_records(r["seasons"])],
        "summary": r["summary"],
        "sensitivity_no_living_cover_months": r["sensitivity"],
        "coverage": r["coverage"],
    }
    (out / "engine1.json").write_text(json.dumps(doc, indent=1, default=str))
    r["smooth"].to_frame().to_parquet(out / "ndvi_smooth.parquet")
sorted(str(p.relative_to(REPO)) for p in PROCESSED.rglob("engine1.json"))

['data/processed/field_a/engine1.json',
 'data/processed/field_b/engine1.json',
 'data/processed/field_c/engine1.json']

## 10 · Engine 2: Stress Recurrence (handoff §8.2)

Sensitive window per season (corn: 14 days around silking estimated at 400 °C capped GDD after Engine 1 green-up; soybean: NDVI peak −14 to +21 days), POWER day-of-year percentiles against 1991–2020, provisional stress rules, relative vigour against this field's own median for the same crop, Scenario A/B/C (§9), and a **base rate**: how often the same calendar window was flagged in 1991–2020, so "k of n" is read against chance. Confidence is capped at moderate (regional drivers). SMAP L4 confirms the dry flags (V11) once its AppEEARS tasks finish.

In [15]:
from pipeline import engine2 as E2
from pipeline.sources import smap as smap_src
smap_dirs = sorted((RAW / "appeears").glob("fieldshift_smap_l4_*")) if (RAW / "appeears").exists() else []
smap_by_field = smap_src.load_points(smap_dirs)
print("SMAP series:", {k: (str(v.index.min().date()), str(v.index.max().date()), len(v)) for k, v in smap_by_field.items()} or "not downloaded yet")
e2_results = {}
for f in F:
    fid = f["id"]
    p = PROCESSED / fid / "engine1.json"
    if not p.exists():
        continue
    e1doc = json.loads(p.read_text())
    st2, summ2 = E2.run(e1doc["seasons"], weather[fid], smap_by_field.get(fid))
    e2_results[fid] = (st2, summ2)
    for crop, d in summ2.items():
        print(f"{fid} {crop}: {d['stress_seasons']} of {d['seasons']} flagged {d['stress_years']}, expected {d['expected_stress_seasons']}, chance of >= observed {d['chance_at_least_observed']}")
if e2_results:
    fid0 = next(iter(e2_results))
    display(pd.DataFrame(E2.to_records(e2_results[fid0][0]))[["season", "crop", "window_start", "window_end", "dry_days", "hot_days", "window_days", "stress_flag", "base_rate", "vigour", "scenario", "smap_agrees", "confidence"]])

SMAP series: not downloaded yet
field_a all_seasons: 4 of 10 flagged [2017, 2021, 2022, 2023], expected 2.9, chance of >= observed 0.315
field_a corn: 1 of 5 flagged [2022], expected 1.4, chance of >= observed 0.817
field_a soybeans: 3 of 5 flagged [2017, 2021, 2023], expected 1.4, chance of >= observed 0.145


field_b all_seasons: 4 of 10 flagged [2017, 2021, 2022, 2023], expected 2.7, chance of >= observed 0.264
field_b corn: 3 of 5 flagged [2017, 2021, 2023], expected 1.3, chance of >= observed 0.121
field_b soybeans: 1 of 5 flagged [2022], expected 1.3, chance of >= observed 0.789
field_c all_seasons: 5 of 10 flagged [2017, 2020, 2021, 2022, 2023], expected 3.0, chance of >= observed 0.15
field_c corn: 3 of 5 flagged [2017, 2021, 2023], expected 1.5, chance of >= observed 0.154
field_c soybeans: 2 of 5 flagged [2020, 2022], expected 1.5, chance of >= observed 0.485


,season,crop,window_start,window_end,dry_days,hot_days,window_days,stress_flag,base_rate,vigour,scenario,smap_agrees,confidence
0,2016,corn,2016-07-10,2016-07-23,0,2,14,False,0.200,uncertain,no stress signal,None,moderate
1,2017,soybeans,2017-07-25,2017-08-29,26,5,36,True,0.300,similar,"B: regional stress, field vigour held (buffered)",None,moderate
2,2018,corn,2018-07-11,2018-07-24,1,3,14,False,0.267,similar,no stress signal,None,moderate
3,2019,soybeans,2019-08-03,2019-09-07,0,0,36,False,0.300,below,C: lower vigour without a stress signal (field...,None,moderate
4,2020,corn,2020-07-22,2020-08-04,0,1,14,False,0.333,below,C: lower vigour without a stress signal (field...,None,moderate
5,2021,soybeans,2021-06-29,2021-08-03,34,13,36,True,0.233,similar,"B: regional stress, field vigour held (buffered)",None,moderate
6,2022,corn,2022-07-18,2022-07-31,14,8,14,True,0.300,similar,"B: regional stress, field vigour held (buffered)",None,moderate
7,2023,soybeans,2023-08-03,2023-09-07,33,18,36,True,0.300,similar,"B: regional stress, field vigour held (buffered)",None,moderate
8,2024,corn,2024-07-25,2024-08-07,0,2,14,False,0.333,similar,no stress signal,None,moderate
9,2025,soybeans,2025-07-15,2025-08-19,0,4,36,False,0.300,similar,no stress signal,None,moderate


In [16]:
for fid, (st2, summ2) in e2_results.items():
    doc = {
        "field_id": fid, "generated": STAMP, "engine": "Engine 2: Stress Recurrence (handoff §8.2)",
        "source": DATASETS["power"] + "; " + DATASETS["power_gwetroot"] + (("; " + DATASETS["smap_l4"]) if fid in smap_by_field else "") + "; timing from Engine 1 (NASA HLS)",
        "method": {"silk_gdd_from_greenup_c": E2.SILK_GDD_FROM_GREENUP, "corn_window_days": E2.CORN_WINDOW_DAYS, "soy_window_days_from_peak": list(E2.SOY_WINDOW),
                   "dry_rule": f"root-zone wetness < p{E2.DRY_PCT:.0f} on >= {E2.DRY_SHARE:.0%} of window days",
                   "hot_rule": f"Tmax > p{E2.HOT_PCT:.0f} on >= {E2.HOT_SHARE:.0%} of window days",
                   "percentile_baseline": list(CLIMATE_BASELINE) if "CLIMATE_BASELINE" in globals() else [1991, 2020], "pool_half_days": E2.POOL_HALF_DAYS,
                   "vigour": f"peak NDVI +/-{E2.VIGOUR_BAND} or integrated NDVI +/-{E2.INTEGRATED_BAND:.0%} vs field median for the crop",
                   "confidence_cap": "moderate (regional drivers)"},
        "seasons": E2.to_records(st2),
        "summary": summ2,
    }
    (PROCESSED / fid / "engine2.json").write_text(json.dumps(doc, indent=1, default=str))
sorted(str(p.relative_to(REPO)) for p in PROCESSED.rglob("engine2.json"))

['data/processed/field_a/engine2.json',
 'data/processed/field_b/engine2.json',
 'data/processed/field_c/engine2.json']

## 11 · Engine 3: Rotation Comparison (handoff §8.3)

Ordinal ratings (better / similar / worse) of each candidate against the current rotation, with evidence, confidence and what to verify. The cover-crop estimate uses **local evidence**: no-living-cover share in cover years with autumn green versus matched years without, across the demo fields (same region). Priorities and constraints are applied live in the app; they never change a rating (V9).

In [17]:
from pipeline import engine3 as E3
e1_docs = {f["id"]: json.loads((PROCESSED / f["id"] / "engine1.json").read_text()) for f in F if (PROCESSED / f["id"] / "engine1.json").exists()}
cover_effect = E3.observed_cover_effect(e1_docs)
print("Local cover effect:", cover_effect)
for f in F:
    fid = f["id"]
    p2 = PROCESSED / fid / "engine2.json"
    if fid not in e1_docs:
        continue
    st_doc = json.loads((PROCESSED / fid / "field_static.json").read_text())
    e2doc = json.loads(p2.read_text()) if p2.exists() else None
    res = E3.rate_field(st_doc, e1_docs[fid], e2doc, cover_effect)
    doc = {"field_id": fid, "generated": STAMP, "engine": "Engine 3: Rotation Comparison (handoff §8.3)",
           "candidates": [{k: (sorted(v) if isinstance(v, set) else v) for k, v in c.items()} for c in E3.CANDIDATES],
           "criteria": E3.CRITERIA, "priorities": E3.PRIORITIES, "criteria_for_priority": E3.CRITERIA_FOR_PRIORITY, "constraints": E3.CONSTRAINTS,
           **res}
    (PROCESSED / fid / "engine3.json").write_text(json.dumps(doc, indent=1, default=str))
    print(fid, {cid: {k: v["rating"] for k, v in crit.items()} for cid, crit in res["ratings"].items() if cid != "cs"})
    print("   rye estimate:", res["ratings"]["cs_rye"]["Soil cover and living roots"]["estimate"], "| erosion importance:", res["erosion_importance"], "| notes:", res["notes"])

Local cover effect: {'share_reduction': 0.14, 'months_per_year': 1.7, 'evidence': {'soybeans': {'with': [('field_a', 2017), ('field_b', 2018), ('field_b', 2022), ('field_b', 2024)], 'n_without': 9}}}
field_a {'cs_rye': {'Soil cover and living roots': 'better', 'Erosion protection': 'better', 'Water and stress exposure': 'similar', 'Nitrogen needs': 'similar', 'Crop diversity': 'better', 'Workload and complexity': 'worse'}, 'cso_rc': {'Soil cover and living roots': 'better', 'Erosion protection': 'better', 'Water and stress exposure': 'better', 'Nitrogen needs': 'better', 'Crop diversity': 'better', 'Workload and complexity': 'worse'}}
   rye estimate: ~7.0 months a year without living cover (now ~8.5) | erosion importance: lower | notes: []
field_b {'cs_rye': {'Soil cover and living roots': 'better', 'Erosion protection': 'better', 'Water and stress exposure': 'similar', 'Nitrogen needs': 'similar', 'Crop diversity': 'better', 'Workload and complexity': 'worse'}, 'cso_rc': {'Soil cover